# DSFB Gray — Crates.io Audit Notebook

Type any crate name from [crates.io](https://crates.io) into the field in **Step 2**, press **Run audit**, and this notebook will:

1. Fetch the latest published source tarball of that crate.
2. Run the deterministic `dsfb-scan-crate` static scanner against it.
3. Display the rendered audit report inline.
4. Package every artifact (plain-text report, SARIF, in-toto statement, DSSE envelope, original crate source) into a single ZIP and auto-trigger a browser download.

The notebook is read-only with respect to the target crate. Nothing is modified in the published crate source; only `dsfb-gray`'s own scanner output directory and an ephemeral scratch staging directory are written to.

---

## What the audit actually does

`dsfb-gray` is the **DSFB Structural Semiotics Engine**: a deterministic, non-intrusive, read-only observer for Rust systems. In this notebook we run only its **static-crate audit path** (the `dsfb-scan-crate` binary). The runtime observer is not exercised here.

### Residual sign triple

At the core of DSFB is the residual sign $(r_k, \omega_k, \alpha_k)$ observed at discrete step $k$:

$$r_k = \text{scalar telemetry residual (any source)}$$
$$\omega_k \approx \text{least-squares slope of } r \text{ over persistence window } P \text{ (drift)}$$
$$\alpha_k \approx \text{difference of half-window slopes of } r \text{ (slew, i.e. curvature)}$$

The static-crate audit does not compute $r$, $\omega$, $\alpha$ — those belong to the runtime path. What the static audit **does** do is match **source-visible structural motifs** that, at runtime, are empirically associated with non-trivial drift and slew signatures. Each motif carries a mapping to a typed `ReasonCode` that the runtime observer would emit if that motif were the proximate cause of a gray-failure transition.

### Admissibility envelope and grammar states

The runtime observer classifies each sign against a regime-conditioned admissibility envelope

$$\mathcal{E}(k) = \{(r, \omega, \alpha) \,:\, \epsilon_r^- \leq r \leq \epsilon_r^+,\; |\omega| \leq \epsilon_\omega,\; |\alpha| \leq \epsilon_\alpha\}$$

and advances a three-state grammar with hysteresis $H$:

$$\text{Admissible} \;\longrightarrow\; \text{Boundary} \;\longrightarrow\; \text{Violation}$$

Transitions in either direction require $\geq H$ consecutive consistent observations, which suppresses chatter on the classification boundary. The static-crate audit does not itself evolve this state machine; it surfaces the **surface-visible evidence** that a reviewer would want to inspect *before* trusting any particular calibration of it.

### What the static audit outputs

For the scanned crate you will get:

- **Overall score** (`dsfb-assurance-score-v1`): a weighted aggregate of source-visible checks — `#![forbid(unsafe_code)]` presence, `no_std` declaration, Power-of-Ten-style discipline, structural heuristic motif hits, evidence of runtime priors, attestation export surface, etc. This is **not** a compliance or certification result.
- **Findings** partitioned into: `defect-candidate`, `design-review`, `review-readiness`, `context-needed`.
- **Heuristic motif hits** from the 12-entry bank — each hit is pinned to file:line evidence inside the crate.
- **Conclusion lenses** — maintainer / compliance-readiness / certification-prep / distributed-operational views over the same evidence set.
- **SARIF 2.1.0** for IDE and CI ingestion.
- **in-toto v1 statement** and **DSSE envelope** (envelope is unsigned unless `DSFB_SCAN_SIGNING_KEY` is set).

### What the audit is explicitly *not*

- Not a compliance certificate (IEC / ISO / RTCA / MIL / NIST).
- Not a proxy for dynamic program analysis or runtime gray-failure detection.
- Not tuned per-crate — the scoring method and heuristic bank are the same for every crate you scan.

---

## How to use this notebook

1. Run **Step 1: Setup** once per Colab session — it installs Rust (if needed), clones `dsfb-gray` fresh, and builds the scanner binary in release mode.
2. In **Step 2: Audit**, type a crate name (e.g. `tokio`, `serde`, `hyper`, `base64`). Leave version blank for the latest stable version, or pin one explicitly.
3. Click **Run audit**. Watch the status log. When it finishes, the report prints inline and your browser downloads a ZIP of every artifact.
4. Step 3 is optional and only useful if you intend to regenerate `dsfb-gray`'s own repo-facing public artifacts.


## Step 1 — Colab setup (run once per session)

Installs Rust (if missing), clones the `dsfb` monorepo (`crates/dsfb-gray/` inside it) from `main` into a fresh workspace, and builds the `dsfb-scan-crate` binary in release mode. The workspace is wiped on every run so you get a clean build every session.


In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from IPython.display import HTML, display

WORKSPACE = Path('/content/dsfb_gray_colab')
CLONE_DIR = WORKSPACE / 'dsfb'
REPO_DIR = CLONE_DIR / 'crates' / 'dsfb-gray'
TARGET_DIR = WORKSPACE / 'targets'
RUNS_DIR = WORKSPACE / 'runs'
ARTIFACTS_DIR = WORKSPACE / 'artifacts'

REPO_URL = 'https://github.com/infinityabundance/dsfb.git'
REPO_REF = 'main'


def status(msg, kind='info'):
    icons = {'info': '⏳', 'ok': '✅', 'warn': '⚠️', 'err': '❌', 'step': '▶'}
    colors = {'info': '#3b82f6', 'ok': '#10b981', 'warn': '#f59e0b', 'err': '#ef4444', 'step': '#6366f1'}
    display(HTML(
        f'<div style="font-family:ui-monospace,monospace;font-size:13px;'
        f'padding:4px 8px;border-left:3px solid {colors[kind]};margin:2px 0;">'
        f'<span style="color:{colors[kind]};">{icons[kind]}</span> {msg}</div>'
    ))


def run(cmd, cwd=None, env=None):
    print('$ ' + ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, env=env, check=True)


t0 = time.time()

status('Step 1/4 — Preparing workspace (wiping any previous session state)', 'step')
for path in [CLONE_DIR, TARGET_DIR, RUNS_DIR, ARTIFACTS_DIR]:
    if path.exists():
        shutil.rmtree(path)
for path in [WORKSPACE, TARGET_DIR, RUNS_DIR, ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
status(f'Workspace ready at {WORKSPACE}', 'ok')

status('Step 2/4 — Checking Rust toolchain', 'step')
if not shutil.which('cargo'):
    status('cargo not found — installing rustup (this takes ~60s)', 'info')
    subprocess.run(['bash', '-lc', 'curl https://sh.rustup.rs -sSf | sh -s -- -y --default-toolchain stable'], check=True)
    status('Rust toolchain installed', 'ok')
else:
    status('Rust toolchain already present', 'ok')

cargo_env = os.environ.copy()
cargo_env['PATH'] = f"{Path.home() / '.cargo' / 'bin'}:{cargo_env['PATH']}"

status(f'Step 3/4 — Cloning dsfb monorepo @ {REPO_REF} (dsfb-gray lives at crates/dsfb-gray/)', 'step')
run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CLONE_DIR)], env=cargo_env)
if not REPO_DIR.is_dir():
    status(f'Expected crate path not found: {REPO_DIR}', 'err')
    raise FileNotFoundError(REPO_DIR)
status(f'Repository cloned; dsfb-gray crate at {REPO_DIR}', 'ok')

status('Step 4/4 — Building dsfb-scan-crate (release profile, 2–5 min on cold cache)', 'step')
run(['cargo', 'build', '--release', '--bin', 'dsfb-scan-crate'], cwd=REPO_DIR, env=cargo_env)
binary_path = REPO_DIR / 'target' / 'release' / 'dsfb-scan-crate'
if not binary_path.exists():
    status(f'Build reported success but binary not found at {binary_path}', 'err')
    raise FileNotFoundError(binary_path)
status(f'Binary ready at {binary_path}', 'ok')

elapsed = time.time() - t0
status(f'Setup complete in {elapsed:.0f}s — proceed to Step 2', 'ok')


## Step 2 — Audit a crates.io crate

Enter a crate name (e.g. `tokio`, `serde`, `base64`, `hyper`). Leave the version field blank to use the latest published stable version, or pin a specific version (e.g. `1.35.0`). Press **Run audit**.

A live status log shows each step. When the audit completes the full report is rendered below and a ZIP of every artifact is automatically downloaded to your browser.


In [ ]:
import io
import json
import os
import re
import shutil
import subprocess
import tarfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from IPython.display import HTML, Markdown, clear_output, display
import ipywidgets as widgets

cargo_env = os.environ.copy()
cargo_env['PATH'] = f"{Path.home() / '.cargo' / 'bin'}:{cargo_env['PATH']}"

CRATES_API = 'https://crates.io/api/v1/crates/{name}'
CRATE_DL = 'https://crates.io/api/v1/crates/{name}/{version}/download'
USER_AGENT = 'dsfb-gray-colab/1.0 (+https://crates.io/crates/dsfb-gray)'
BINARY = REPO_DIR / 'target' / 'release' / 'dsfb-scan-crate'


def http_get_json(url: str) -> dict:
    req = Request(url, headers={'User-Agent': USER_AGENT})
    with urlopen(req, timeout=30) as resp:
        return json.load(resp)


def http_get_bytes(url: str) -> bytes:
    req = Request(url, headers={'User-Agent': USER_AGENT})
    with urlopen(req, timeout=120) as resp:
        return resp.read()


def resolve_version(crate_name: str, version_text: str) -> tuple[str, str]:
    version_text = (version_text or '').strip()
    data = http_get_json(CRATES_API.format(name=crate_name))
    crate = data.get('crate', {})
    latest_stable = crate.get('max_stable_version') or crate.get('max_version')
    total_downloads = crate.get('downloads', 0)
    if version_text:
        return version_text, f'pinned (latest stable is {latest_stable}, {total_downloads:,} total downloads)'
    if not latest_stable:
        raise RuntimeError(f'Could not resolve latest version for crate {crate_name!r}.')
    return latest_stable, f'{total_downloads:,} total downloads'


def download_and_extract_crate(crate_name: str, version: str, dest_root: Path, logger) -> tuple[Path, int]:
    dest_root.mkdir(parents=True, exist_ok=True)
    archive_path = dest_root / f'{crate_name}-{version}.crate'

    url = CRATE_DL.format(name=crate_name, version=version)
    logger(f'GET {url}', 'info')
    crate_bytes = http_get_bytes(url)
    archive_path.write_bytes(crate_bytes)
    size_kb = len(crate_bytes) // 1024

    with tarfile.open(archive_path, mode='r:gz') as tar:
        try:
            tar.extractall(path=dest_root, filter='data')
        except TypeError:
            tar.extractall(path=dest_root)

    extracted = dest_root / f'{crate_name}-{version}'
    if not extracted.is_dir():
        subdirs = [p for p in dest_root.iterdir() if p.is_dir()]
        if len(subdirs) == 1:
            extracted = subdirs[0]
        else:
            raise RuntimeError(f'Could not locate extracted crate directory under {dest_root}.')
    return extracted, size_kb


def find_latest_run_dir(out_dir: Path) -> Path:
    run_dirs = sorted([p for p in out_dir.iterdir() if p.is_dir() and p.name.startswith('dsfb-gray-')])
    if not run_dirs:
        raise RuntimeError(f'No scan run directory created under {out_dir}.')
    return run_dirs[-1]


def find_scan_files(run_dir: Path, crate_name: str) -> dict:
    stem = re.sub(r'[^A-Za-z0-9_]', '_', crate_name) + '_scan'
    return {
        'report': run_dir / f'{stem}.txt',
        'sarif':  run_dir / f'{stem}.sarif.json',
        'intoto': run_dir / f'{stem}.intoto.json',
        'dsse':   run_dir / f'{stem}.dsse.json',
    }


def read_text_safe(path: Path, default='') -> str:
    if not path.exists():
        return default
    return path.read_text(encoding='utf-8', errors='replace')


def zip_artifacts(run_dir: Path, crate_source_dir: Path, zip_path: Path) -> int:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(run_dir.rglob('*')):
            if file_path.is_file():
                zf.write(file_path, arcname=f'artifacts/{file_path.relative_to(run_dir)}')
        for file_path in sorted(crate_source_dir.rglob('*')):
            if file_path.is_file():
                zf.write(file_path, arcname=f'crate_source/{file_path.relative_to(crate_source_dir)}')
    return zip_path.stat().st_size


class Logger:
    def __init__(self, out_widget):
        self.out = out_widget
    def __call__(self, msg, kind='info'):
        icons = {'info': '⏳', 'ok': '✅', 'warn': '⚠️', 'err': '❌', 'step': '▶'}
        colors = {'info': '#3b82f6', 'ok': '#10b981', 'warn': '#f59e0b', 'err': '#ef4444', 'step': '#6366f1'}
        with self.out:
            display(HTML(
                f'<div style="font-family:ui-monospace,monospace;font-size:13px;'
                f'padding:3px 8px;border-left:3px solid {colors[kind]};margin:2px 0;">'
                f'<span style="color:{colors[kind]};font-weight:600;">{icons[kind]}</span> {msg}</div>'
            ))


def run_audit(crate_name: str, version_text: str, logger):
    crate_name = crate_name.strip()
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_\-]{0,63}', crate_name):
        raise ValueError(f'Invalid crates.io crate name: {crate_name!r}')

    t0 = time.time()
    logger(f'Step 1/6 — Resolving <b>{crate_name}</b> on crates.io', 'step')
    version, meta = resolve_version(crate_name, version_text)
    logger(f'Resolved version <b>{version}</b> — {meta}', 'ok')

    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    staging = TARGET_DIR / f'{crate_name}-{version}-{stamp}'
    out_root = RUNS_DIR / f'{crate_name}-{version}-{stamp}'
    out_root.mkdir(parents=True, exist_ok=True)

    logger(f'Step 2/6 — Downloading <b>{crate_name} {version}</b> tarball', 'step')
    crate_source_dir, size_kb = download_and_extract_crate(crate_name, version, staging, logger)
    logger(f'Downloaded {size_kb} KB, extracted to <code>{crate_source_dir}</code>', 'ok')

    logger('Step 3/6 — Running dsfb-scan-crate (deterministic static audit)', 'step')
    cmd = [str(BINARY), '--out-dir', str(out_root), str(crate_source_dir)]
    scan_t = time.time()
    proc = subprocess.run(cmd, cwd=REPO_DIR, env=cargo_env, text=True, capture_output=True, check=False)
    scan_elapsed = time.time() - scan_t
    if proc.returncode != 0:
        logger(f'Scanner exited with code {proc.returncode}', 'err')
        raise RuntimeError('DSFB scan failed.\n\nSTDOUT:\n' + proc.stdout + '\n\nSTDERR:\n' + proc.stderr)
    logger(f'Scanner finished in {scan_elapsed:.1f}s', 'ok')

    logger('Step 4/6 — Locating generated artifacts', 'step')
    run_dir = find_latest_run_dir(out_root)
    files = find_scan_files(run_dir, crate_name)
    missing = [k for k, v in files.items() if not v.exists()]
    if missing:
        logger(f'Missing expected artifact(s): {missing}', 'warn')
    logger(f'Run directory: <code>{run_dir}</code>', 'ok')

    logger('Step 5/6 — Packaging ZIP bundle (report + SARIF + in-toto + DSSE + crate source)', 'step')
    zip_path = ARTIFACTS_DIR / f'dsfb_audit_{crate_name}_{version}_{stamp}.zip'
    zip_size = zip_artifacts(run_dir, crate_source_dir, zip_path)
    logger(f'ZIP written: {zip_path.name} ({zip_size // 1024} KB)', 'ok')

    logger('Step 6/6 — Preparing download + inline report', 'step')
    total_elapsed = time.time() - t0
    logger(f'Audit pipeline complete in {total_elapsed:.1f}s', 'ok')

    return {
        'crate_name': crate_name, 'version': version,
        'crate_source_dir': crate_source_dir, 'run_dir': run_dir,
        'files': files, 'zip_path': zip_path,
        'stdout': proc.stdout, 'stderr': proc.stderr,
        'elapsed': total_elapsed,
    }


# --- Widget UI ---
crate_name_widget = widgets.Text(
    value='base64', description='Crate:', placeholder='e.g. tokio, serde, hyper',
    layout=widgets.Layout(width='460px'),
)
version_widget = widgets.Text(
    value='', description='Version:', placeholder='blank = latest stable',
    layout=widgets.Layout(width='460px'),
)
run_button = widgets.Button(description='▶ Run audit', button_style='primary',
                             layout=widgets.Layout(width='160px'))

status_out = widgets.Output()
result_out = widgets.Output()


def on_run_clicked(_):
    run_button.disabled = True
    run_button.description = 'Running…'
    with status_out: clear_output()
    with result_out: clear_output()
    logger = Logger(status_out)
    try:
        result = run_audit(crate_name_widget.value, version_widget.value, logger)
    except Exception as exc:
        logger(f'Audit aborted: {exc}', 'err')
        run_button.disabled = False
        run_button.description = '▶ Run audit'
        return

    with result_out:
        display(Markdown(
            f"## Audit summary\n"
            f"- **Crate:** `{result['crate_name']}` @ **{result['version']}**\n"
            f"- **Run dir:** `{result['run_dir']}`\n"
            f"- **ZIP:** `{result['zip_path'].name}` — auto-downloading\n"
            f"- **Wall time:** {result['elapsed']:.1f}s\n"
        ))

        display(Markdown('### Scanner status (stderr)'))
        print(result['stderr'] or '[no stderr output]')

        display(Markdown('### Full audit report'))
        report_text = read_text_safe(result['files']['report'])
        max_inline = 60000
        if len(report_text) > max_inline:
            print(report_text[:max_inline])
            print(f'\n[report truncated at {max_inline} chars — full text in the ZIP]')
        else:
            print(report_text)

        try:
            from google.colab import files as colab_files
            colab_files.download(str(result['zip_path']))
            display(HTML(
                f'<div style="padding:8px 12px;background:#ecfdf5;border:1px solid #10b981;border-radius:4px;margin-top:12px;">'
                f'⬇️ Your browser is downloading <b>{result["zip_path"].name}</b>. '
                f'If the download did not start, use the file browser on the left to grab it from '
                f'<code>{result["zip_path"]}</code>.</div>'
            ))
        except ImportError:
            display(HTML(
                f'<div style="padding:8px 12px;background:#fffbeb;border:1px solid #f59e0b;border-radius:4px;margin-top:12px;">'
                f'Not running in Colab — grab the ZIP from <code>{result["zip_path"]}</code>.</div>'
            ))

    run_button.disabled = False
    run_button.description = '▶ Run audit'


run_button.on_click(on_run_clicked)
display(widgets.VBox([
    widgets.HTML('<h3 style="margin:0;">DSFB crates.io audit</h3>'),
    crate_name_widget, version_widget, run_button,
    widgets.HTML('<hr style="margin:8px 0;"/>'),
    widgets.HTML('<div style="font-size:12px;color:#6b7280;">Status log</div>'),
    status_out,
    widgets.HTML('<hr style="margin:8px 0;"/>'),
    result_out,
]))


## Step 3 — Optional: regenerate `dsfb-gray`'s own repo public artifacts

Only useful if you intend to refresh the checked-in `paper/generated/` fragments for the `dsfb-gray` repository itself. Not required to audit other crates.


In [ ]:
import subprocess

proc = subprocess.run(
    ['cargo', 'run', '--release', '--bin', 'dsfb-regenerate-public-artifacts'],
    cwd=REPO_DIR, env=cargo_env, text=True, capture_output=True, check=False,
)
print(proc.stdout)
print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError('Public artifact regeneration failed.')
